# yeast3: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('03_yeast3_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,83.397,75.911,81.954,71.136,98.032,84.584,82.849,97.174,79.654
smote,88.755,75.473,69.861,82.547,95.496,89.022,81.187,96.865,82.114
bsmote,89.480,70.797,60.137,86.222,92.884,89.553,78.426,96.274,80.139
ros,89.677,74.800,67.152,85.000,94.663,89.831,81.275,96.806,82.238
adasyn,89.822,71.325,60.694,86.837,92.923,89.880,79.844,96.544,80.573
casb,89.383,75.177,68.202,84.081,95.080,89.581,81.183,96.806,82.280
casb_relaxed,89.422,76.311,70.340,83.750,95.534,89.642,81.186,96.880,82.867
casb_edited,89.664,73.208,64.246,85.634,93.944,89.789,81.549,96.818,81.436
casb_ros,89.674,74.705,66.994,85.009,94.664,89.837,81.222,96.710,82.190


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,83.050,75.136,80.490,70.530,97.881,84.206,76.084,93.119,79.093
smote,88.132,70.195,61.002,83.400,93.224,88.312,72.796,93.626,79.164
bsmote,88.756,67.569,55.697,86.165,91.446,88.805,63.538,93.002,78.162
ros,88.682,69.563,59.076,84.934,92.657,88.795,70.739,92.687,79.123
adasyn,88.949,65.855,52.816,87.689,90.272,88.981,66.309,93.030,77.402
casb,88.963,70.924,61.057,84.934,93.225,89.079,72.771,93.653,79.943
casb_relaxed,90.156,73.181,63.498,86.771,93.718,90.244,74.309,93.663,81.669
casb_edited,90.102,71.549,60.733,87.396,92.922,90.159,71.492,93.840,80.826
casb_ros,88.597,70.383,60.573,84.318,93.148,88.733,70.738,92.764,79.490


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,82.214,73.887,79.675,69.318,97.767,83.543,83.870,96.909,78.051
smote,89.424,78.786,75.180,82.869,96.593,89.731,83.441,96.957,84.105
bsmote,90.241,77.893,71.960,85.000,95.874,90.437,81.064,96.799,84.067
ros,89.159,78.078,74.186,82.500,96.442,89.471,83.735,97.120,83.618
adasyn,90.341,76.943,70.001,85.606,95.420,90.513,81.746,96.837,83.642
casb,88.896,77.990,74.524,81.941,96.518,89.230,82.906,97.029,83.443
casb_relaxed,89.200,78.187,74.296,82.557,96.480,89.518,83.939,97.029,83.693
casb_edited,90.946,78.095,71.359,86.534,95.609,91.072,84.716,97.099,84.520
casb_ros,89.993,78.801,74.218,84.072,96.366,90.219,83.717,97.019,84.397


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,80.570,72.485,80.503,66.761,97.956,82.359,79.046,96.729,76.527
smote,87.616,71.045,63.456,81.903,93.868,87.886,78.858,96.704,79.331
bsmote,89.026,67.621,55.718,87.102,91.066,89.084,78.232,96.586,78.324
ros,88.632,70.852,61.731,84.375,93.226,88.800,79.015,96.745,79.742
adasyn,88.909,67.637,55.989,86.828,91.105,88.966,78.534,96.625,78.273
casb,87.661,72.002,65.323,81.610,94.285,87.947,78.496,96.679,79.831
casb_relaxed,87.713,72.289,65.787,81.610,94.398,88.004,78.481,96.637,80.001
casb_edited,89.036,69.658,59.027,85.890,92.354,89.122,78.542,96.630,79.347
casb_ros,88.024,70.860,62.972,83.153,93.377,88.265,78.416,96.660,79.442


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,82.308,74.355,80.655,69.437,97.909,83.673,80.462,95.983,78.331
smote,88.482,73.875,67.375,82.680,94.795,88.738,79.070,96.038,81.178
bsmote,89.376,70.970,60.878,86.122,92.818,89.470,75.315,95.665,80.173
ros,89.037,73.323,65.536,84.202,94.247,89.224,78.691,95.840,81.180
adasyn,89.505,70.440,59.875,86.740,92.430,89.585,76.608,95.759,79.972
casb,88.726,74.023,67.277,83.142,94.777,88.959,78.839,96.042,81.374
casb_relaxed,89.123,74.992,68.480,83.672,95.032,89.352,79.479,96.053,82.057
casb_edited,89.937,73.128,63.841,86.364,93.708,90.036,79.075,96.097,81.532
casb_ros,89.072,73.687,66.189,84.138,94.389,89.264,78.523,95.788,81.380


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,89.674,adasyn,89.822,-0.148,LOSS
1,f1,casb_relaxed,76.311,smote,75.473,0.838,WIN
2,precision,casb_relaxed,70.340,smote,69.861,0.479,WIN
3,recall,casb_edited,85.634,adasyn,86.837,-1.203,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,90.156,adasyn,88.949,1.207,WIN
1,f1,casb_relaxed,73.181,smote,70.195,2.986,WIN
2,precision,casb_relaxed,63.498,smote,61.002,2.496,WIN
3,recall,casb_edited,87.396,adasyn,87.689,-0.294,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,90.946,adasyn,90.341,0.605,WIN
1,f1,casb_ros,78.801,smote,78.786,0.015,WIN
2,precision,casb,74.524,smote,75.180,-0.656,LOSS
3,recall,casb_edited,86.534,adasyn,85.606,0.928,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,89.937,adasyn,89.505,0.432,WIN
1,f1,casb_relaxed,74.992,smote,73.875,1.117,WIN
2,precision,casb_relaxed,68.480,smote,67.375,1.105,WIN
3,recall,casb_edited,86.364,adasyn,86.740,-0.376,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.813398,0.771930,0.916667,0.666667,0.992424,0.829545,0.923057,0.987489,0.792664,None
1,42,1,svm,smote,0.50,0.907068,0.811594,0.777778,0.848485,0.969697,0.909091,0.894871,0.983012,0.859331,None
2,42,1,svm,bsmote,0.75,0.925979,0.769231,0.666667,0.909091,0.943182,0.926136,0.880918,0.977732,0.847605,None
3,42,1,svm,ros,0.50,0.907068,0.811594,0.777778,0.848485,0.969697,0.909091,0.908633,0.984963,0.859331,None
4,42,1,svm,adasyn,0.50,0.914065,0.773333,0.690476,0.878788,0.950758,0.914773,0.896168,0.981749,0.843699,None
5,42,1,svm,casb,0.50,0.890724,0.794118,0.771429,0.818182,0.969697,0.893939,0.894311,0.984389,0.842421,None
6,42,1,svm,casb_relaxed,0.50,0.910605,0.835821,0.823529,0.848485,0.977273,0.912879,0.911465,0.986455,0.873213,None
7,42,1,svm,casb_edited,0.50,0.905295,0.800000,0.756757,0.848485,0.965909,0.907197,0.917162,0.986570,0.852648,None
8,42,1,svm,casb_ros,0.50,0.907068,0.811594,0.777778,0.848485,0.969697,0.909091,0.896703,0.984619,0.859331,None
9,42,1,knn,original,0.50,0.847944,0.800000,0.888889,0.727273,0.988636,0.857955,0.852550,0.938648,0.823972,None
